In [4]:
import pandas as pd
from pathlib import Path

base = Path("../data/external/epic_kitchens/annotations")

train_path = base / "EPIC_100_train.csv"
val_path = base / "EPIC_100_validation.csv"

train = pd.read_csv(train_path)
val = pd.read_csv(val_path)

print(train.shape)
print(train.columns)
print(train.head())

(67217, 15)
Index(['narration_id', 'participant_id', 'video_id', 'narration_timestamp',
       'start_timestamp', 'stop_timestamp', 'start_frame', 'stop_frame',
       'narration', 'verb', 'verb_class', 'noun', 'noun_class', 'all_nouns',
       'all_noun_classes'],
      dtype='object')
  narration_id participant_id video_id narration_timestamp start_timestamp  \
0     P01_01_0            P01   P01_01        00:00:01.089     00:00:00.14   
1     P01_01_1            P01   P01_01        00:00:02.629     00:00:04.37   
2    P01_01_10            P01   P01_01        00:00:23.340     00:00:24.97   
3   P01_01_100            P01   P01_01        00:07:57.919     00:07:59.75   
4   P01_01_101            P01   P01_01        00:08:00.020     00:08:01.47   

  stop_timestamp  start_frame  stop_frame      narration     verb  verb_class  \
0    00:00:03.37            8         202      open door     open           3   
1    00:00:06.17          262         370  turn on light  turn-on           6   


In [5]:
cols = [
    "participant_id",
    "video_id",
    "start_timestamp",
    "stop_timestamp",
    "verb",
    "noun",
    "verb_class",
    "noun_class",
    "all_nouns",
    "all_noun_classes"
]

existing_cols = [c for c in cols if c in train.columns]
print(train[existing_cols].head())

  participant_id video_id start_timestamp stop_timestamp     verb      noun  \
0            P01   P01_01     00:00:00.14    00:00:03.37     open      door   
1            P01   P01_01     00:00:04.37    00:00:06.17  turn-on     light   
2            P01   P01_01     00:00:24.97    00:00:26.20     open    drawer   
3            P01   P01_01     00:07:59.75    00:08:00.88     take       cup   
4            P01   P01_01     00:08:01.47    00:08:02.21     open  cupboard   

   verb_class  noun_class     all_nouns all_noun_classes  
0           3           3      ['door']              [3]  
1           6         114     ['light']            [114]  
2           3           8    ['drawer']              [8]  
3           0          13       ['cup']             [13]  
4           3           3  ['cupboard']              [3]  


In [6]:
print(train["verb"].value_counts().head(30))

verb
pick-up      9868
put-down     7726
open         4851
close        3463
take         3412
rinse        2778
turn-on      2187
put          2106
wash         2078
put-in       2035
turn-off     1806
move         1479
put-on       1164
stir         1123
cut           830
dry           710
put-into      682
clean         657
pour          655
grab          643
lather        598
get           586
shake         527
place         504
throw         501
pour-into     439
remove        393
wipe          359
adjust        353
chop          349
Name: count, dtype: int64


In [10]:
verb_counts = train["verb"].value_counts().reset_index()
verb_counts.columns = ["verb", "count"]
verb_counts.to_csv("../data/annotations/epic_kitchens/verb_counts.csv", index=False)

In [11]:
print(train["noun"].value_counts().head(50))

noun
tap               3567
plate             2186
knife             2092
spoon             1806
cupboard          1786
drawer            1590
glass             1526
sponge            1500
pan               1438
lid               1412
hand              1401
bowl              1335
fridge            1149
fork              1074
onion              860
cloth              828
spatula            824
container          758
dough              743
meat               681
water              631
bottle             618
pot                607
bag                538
salt               524
saucepan           521
mug                505
board:chopping     458
cup                421
kettle             412
oil                395
board:cutting      392
food               386
cheese             380
egg                375
pasta              373
potato             364
carrot             362
door               356
hob                354
colander           347
tomato             311
garlic             311
jar   

In [12]:
target_verbs = [
    "cut", "peel", "pour", "mix", "stir",
    "wash", "take", "put", "open", "close"
]

subset = train[train["verb"].isin(target_verbs)].copy()

print(subset.shape)
print(subset["verb"].value_counts())

(19032, 15)
verb
open     4851
close    3463
take     3412
put      2106
wash     2078
stir     1123
cut       830
pour      655
peel      302
mix       212
Name: count, dtype: int64


In [14]:
subset.to_csv("../data/annotations/epic_kitchens/epic_train_target_verbs.csv", index=False)

In [15]:
val_subset = val[val["verb"].isin(target_verbs)].copy()
val_subset.to_csv("../data/annotations/epic_kitchens/epic_val_target_verbs.csv", index=False)

In [16]:
counts = subset["verb"].value_counts()
print(counts)

verb
open     4851
close    3463
take     3412
put      2106
wash     2078
stir     1123
cut       830
pour      655
peel      302
mix       212
Name: count, dtype: int64


In [17]:
balanced_subset = (
    subset.groupby("verb", group_keys=False)
    .apply(lambda x: x.sample(min(len(x), 200), random_state=42))
    .reset_index(drop=True)
)

balanced_subset["verb"].value_counts()

verb
close    200
cut      200
mix      200
open     200
peel     200
pour     200
put      200
stir     200
take     200
wash     200
Name: count, dtype: int64

In [19]:
balanced_subset.to_csv(
    "../data/annotations/epic_kitchens/epic_train_balanced_200_per_class.csv",
    index=False
)

In [20]:
import pandas as pd
from pathlib import Path

subset_path = Path("../data/annotations/epic_kitchens/epic_train_balanced_200_per_class.csv")

df = pd.read_csv(subset_path)

print("Nombre de segments :", len(df))
print("Colonnes :", df.columns.tolist())

print("\nExemples :")
print(df.head())

print("\nNombre de vidéos différentes :")
print(df["video_id"].nunique())

print("\nTop vidéos utilisées :")
print(df["video_id"].value_counts().head(20))

Nombre de segments : 2000
Colonnes : ['narration_id', 'participant_id', 'video_id', 'narration_timestamp', 'start_timestamp', 'stop_timestamp', 'start_frame', 'stop_frame', 'narration', 'verb', 'verb_class', 'noun', 'noun_class', 'all_nouns', 'all_noun_classes']

Exemples :
  narration_id participant_id video_id narration_timestamp start_timestamp  \
0   P35_105_21            P35  P35_105        00:01:04.697     00:01:04.01   
1   P02_09_220            P02   P02_09        00:13:29.700     00:13:29.57   
2  P35_107_236            P35  P35_107        00:17:49.351     00:17:48.70   
3   P08_05_287            P08   P08_05        00:23:46.340     00:23:45.54   
4  P04_121_660            P04  P04_121        00:26:29.103     00:26:28.20   

  stop_timestamp  start_frame  stop_frame           narration   verb  \
0    00:01:05.83         3200        3291      close cupboard  close   
1    00:13:30.69        48574       48641     close trash bin  close   
2    00:17:49.97        53435       5349

In [21]:
target_verbs = ["cut", "peel", "pour", "wash", "take", "put", "open", "close"]

df_small = df[df["verb"].isin(target_verbs)].copy()

df_small = (
    df_small.groupby("verb", group_keys=False)
    .apply(lambda x: x.sample(min(len(x), 30), random_state=42))
    .reset_index(drop=True)
)

print(df_small["verb"].value_counts())
print("Nombre total de segments :", len(df_small))
print("Nombre de vidéos différentes :", df_small["video_id"].nunique())

df_small.to_csv("../data/annotations/epic_kitchens/epic_mvp_30_per_class.csv", index=False)

verb
close    30
cut      30
open     30
peel     30
pour     30
put      30
take     30
wash     30
Name: count, dtype: int64
Nombre total de segments : 240
Nombre de vidéos différentes : 144


In [22]:
video_list = df_small["video_id"].drop_duplicates().sort_values()

print("Nombre de vidéos à télécharger :", len(video_list))
print(video_list.head(20))

video_list.to_csv(
    "../data/annotations/epic_kitchens/videos_to_download_mvp.txt",
    index=False,
    header=False
)

Nombre de vidéos à télécharger : 144
33      P01_01
28      P01_03
109     P01_09
191    P01_102
145    P01_104
59     P01_105
22     P01_106
4      P01_107
17     P01_108
43     P01_109
144     P01_17
64      P01_18
44      P02_03
158     P02_04
224     P02_06
61      P02_09
127    P02_101
6      P02_102
27     P02_104
39     P02_109
Name: video_id, dtype: object


In [24]:
!python ../data/external/epic_kitchens/download_scripts/epic_downloader.py --help

usage: epic_downloader.py [-h] [--output-path [OUTPUT_PATH]] [--videos]
                          [--rgb-frames] [--flow-frames]
                          [--object-detection-images] [--masks] [--metadata]
                          [--consent-forms] [--participants [PARTICIPANTS]]
                          [--specific-videos [SPECIFIC_VIDEOS]]
                          [--extension-only] [--epic55-only]
                          [--action-recognition] [--domain-adaptation]
                          [--action-retrieval] [--train] [--val] [--test]
                          [--source-train] [--source-test] [--target-train]
                          [--target-test] [--val-source-train]
                          [--val-source-test] [--val-target-train]
                          [--val-target-test] [--errata]

optional arguments:
  -h, --help            show this help message and exit
  --output-path [OUTPUT_PATH]
                        Path where to download files. Default is
             

In [25]:
from pathlib import Path

path = Path("../data/annotations/epic_kitchens/videos_to_download_mvp.txt")

print(path.exists())
print(path.read_text().splitlines()[:20])

True
['P01_01', 'P01_03', 'P01_09', 'P01_102', 'P01_104', 'P01_105', 'P01_106', 'P01_107', 'P01_108', 'P01_109', 'P01_17', 'P01_18', 'P02_03', 'P02_04', 'P02_06', 'P02_09', 'P02_101', 'P02_102', 'P02_104', 'P02_109']


In [2]:
import pandas as pd
from pathlib import Path

video_info_path = Path("C:/Users/Rayen/Documents/GitHub/Stage_2026/data/external/epic_kitchens/annotations/EPIC_100_video_info.csv")

video_info = pd.read_csv(video_info_path)

print(video_info.shape)
print(video_info.columns.tolist())
print(video_info.head())

(700, 4)
['video_id', 'duration', 'fps', 'resolution']
  video_id     duration       fps resolution
0   P01_01  1652.152817  59.94006  1920x1080
1   P01_02   502.134967  59.94006  1920x1080
2   P01_03   118.852067  59.94006  1920x1080
3   P01_04   105.238467  59.94006  1920x1080
4   P01_05  1271.988033  59.94006  1920x1080


In [3]:
for col in video_info.columns:
    print(col)

video_id
duration
fps
resolution


In [5]:
import pandas as pd
from pathlib import Path

# Fichiers
subset_path = Path("../data/annotations/epic_kitchens/epic_mvp_30_per_class.csv")
video_info_path = Path("C:/Users/Rayen/Documents/GitHub/Stage_2026/data/external/epic_kitchens/annotations/EPIC_100_video_info.csv")

# Chargement
df = pd.read_csv(subset_path)
video_info = pd.read_csv(video_info_path)

print("Colonnes video_info :")
print(video_info.columns.tolist())

# Vérification de la colonne video_id
if "video_id" not in video_info.columns:
    raise ValueError("La colonne video_id n'existe pas dans EPIC_100_video_info.csv")

# Détection automatique de la colonne durée
possible_duration_cols = [
    "duration",
    "duration_seconds",
    "duration_secs",
    "video_duration",
    "length",
    "seconds"
]

duration_col = None

for col in possible_duration_cols:
    if col in video_info.columns:
        duration_col = col
        break

print("Colonne durée détectée :", duration_col)

Colonnes video_info :
['video_id', 'duration', 'fps', 'resolution']
Colonne durée détectée : duration


In [6]:
def duration_to_seconds(x):
    """
    Convertit une durée en secondes.
    Accepte :
    - nombre : 245.3
    - texte : '00:04:05'
    - texte : '04:05'
    """
    if pd.isna(x):
        return None
    
    # Si c'est déjà un nombre
    if isinstance(x, (int, float)):
        return float(x)
    
    x = str(x).strip()
    
    # Format HH:MM:SS ou MM:SS
    if ":" in x:
        parts = x.split(":")
        parts = [float(p) for p in parts]
        
        if len(parts) == 3:
            h, m, s = parts
            return h * 3600 + m * 60 + s
        
        if len(parts) == 2:
            m, s = parts
            return m * 60 + s
    
    # Format texte numérique
    try:
        return float(x)
    except:
        return None

In [7]:
# Remplacer ici si la colonne durée n'a pas été détectée automatiquement
# Exemple : duration_col = "duration"
if duration_col is None:
    raise ValueError("Il faut définir manuellement duration_col après avoir vu les colonnes.")

video_info["duration_seconds_clean"] = video_info[duration_col].apply(duration_to_seconds)

# Garder uniquement les vidéos <= 5 minutes = 300 secondes
short_videos = video_info[video_info["duration_seconds_clean"] <= 300].copy()

print("Nombre total de vidéos :", video_info["video_id"].nunique())
print("Nombre de vidéos <= 5 min :", short_videos["video_id"].nunique())

print(short_videos[["video_id", duration_col, "duration_seconds_clean"]].head())

Nombre total de vidéos : 700
Nombre de vidéos <= 5 min : 347
  video_id    duration  duration_seconds_clean
2   P01_03  118.852067              118.852067
3   P01_04  105.238467              105.238467
6   P01_07  161.394567              161.394567
7   P01_08   98.965533               98.965533
9   P01_10  139.038900              139.038900


In [9]:
short_video_ids = set(short_videos["video_id"])

df_short = df[df["video_id"].isin(short_video_ids)].copy()

print("Segments avant filtrage :", len(df))
print("Segments après filtrage vidéos <= 5 min :", len(df_short))

print("\nNombre de vidéos restantes :", df_short["video_id"].nunique())

print("\nDistribution des verbes après filtrage :")
print(df_short["verb"].value_counts())

Segments avant filtrage : 240
Segments après filtrage vidéos <= 5 min : 30

Nombre de vidéos restantes : 23

Distribution des verbes après filtrage :
verb
close    10
put       6
open      4
pour      4
wash      4
cut       1
take      1
Name: count, dtype: int64


In [10]:
df_short_balanced = (
    df_short.groupby("verb", group_keys=False)
    .apply(lambda x: x.sample(min(len(x), 30), random_state=42))
    .reset_index(drop=True)
)

print("Distribution finale :")
print(df_short_balanced["verb"].value_counts())

print("\nNombre total de segments :", len(df_short_balanced))
print("Nombre de vidéos différentes :", df_short_balanced["video_id"].nunique())

Distribution finale :
verb
close    10
put       6
open      4
pour      4
wash      4
cut       1
take      1
Name: count, dtype: int64

Nombre total de segments : 30
Nombre de vidéos différentes : 23


In [11]:
output_csv = Path("../data/annotations/epic_kitchens/epic_mvp_30_per_class_max5min.csv")

df_short_balanced.to_csv(output_csv, index=False)

print("Fichier sauvegardé :", output_csv)

Fichier sauvegardé : ..\data\annotations\epic_kitchens\epic_mvp_30_per_class_max5min.csv


In [13]:
video_list = df_short_balanced["video_id"].drop_duplicates().sort_values()

output_txt = Path("../data/annotations/epic_kitchens/videos_to_download_mvp_max5min.txt")

video_list.to_csv(output_txt, index=False, header=False)

print("Nombre de vidéos à télécharger :", len(video_list))
print("Fichier sauvegardé :", output_txt)
print(video_list.head(20).tolist())

Nombre de vidéos à télécharger : 23
Fichier sauvegardé : ..\data\annotations\epic_kitchens\videos_to_download_mvp_max5min.txt
['P01_03', 'P01_102', 'P01_104', 'P01_107', 'P01_108', 'P02_101', 'P02_102', 'P02_133', 'P02_135', 'P03_14', 'P03_16', 'P04_16', 'P07_01', 'P07_05', 'P07_107', 'P08_02', 'P09_05', 'P11_101', 'P11_109', 'P11_11']


In [14]:
from pathlib import Path

video_list_path = Path("../data/annotations/epic_kitchens/videos_to_download_mvp_max5min.txt")

video_ids = video_list_path.read_text().splitlines()
video_ids = [v.strip() for v in video_ids if v.strip()]

videos_as_text = " ".join(video_ids)

output_path = r"C:\Users\Rayen\Documents\GitHub\Stage_2026\data\epic_videos"

command = f'python epic_downloader.py --videos --specific-videos {videos_as_text} --output-path "{output_path}"'

print(command)

python epic_downloader.py --videos --specific-videos P01_03 P01_102 P01_104 P01_107 P01_108 P02_101 P02_102 P02_133 P02_135 P03_14 P03_16 P04_16 P07_01 P07_05 P07_107 P08_02 P09_05 P11_101 P11_109 P11_11 P22_102 P25_03 P26_110 --output-path "C:\Users\Rayen\Documents\GitHub\Stage_2026\data\epic_videos"


In [ ]:
!python epic_downloader.py --videos --specific-videos P01_03 P01_102 P01_104 P01_107 P01_108 P02_101 P02_102 P02_133 P02_135 P03_14 P03_16 P04_16 P07_01 P07_05 P07_107 P08_02 P09_05 P11_101 P11_109 P11_11 P22_102 P25_03 P26_110 --output-path "C:\Users\Rayen\Documents\GitHub\Stage_2026\data\epic_videos"

In [15]:
from pathlib import Path

video_list_path = Path("../data/annotations/epic_kitchens/videos_to_download_mvp_max5min.txt")

video_ids = video_list_path.read_text().splitlines()
video_ids = [v.strip() for v in video_ids if v.strip()]

output_path = r"C:\Users\Rayen\Documents\GitHub\Stage_2026\data\epic_videos"

batch_size = 20

commands = []

for i in range(0, len(video_ids), batch_size):
    batch = video_ids[i:i + batch_size]
    videos_as_text = " ".join(batch)
    cmd = f'python epic_downloader.py --videos --specific-videos {videos_as_text} --output-path "{output_path}"'
    commands.append(cmd)

for idx, cmd in enumerate(commands, start=1):
    print(f"\n# Batch {idx}")
    print(cmd)


# Batch 1
python epic_downloader.py --videos --specific-videos P01_03 P01_102 P01_104 P01_107 P01_108 P02_101 P02_102 P02_133 P02_135 P03_14 P03_16 P04_16 P07_01 P07_05 P07_107 P08_02 P09_05 P11_101 P11_109 P11_11 --output-path "C:\Users\Rayen\Documents\GitHub\Stage_2026\data\epic_videos"

# Batch 2
python epic_downloader.py --videos --specific-videos P22_102 P25_03 P26_110 --output-path "C:\Users\Rayen\Documents\GitHub\Stage_2026\data\epic_videos"
